In [ ]:
import os
import re

from glob import glob
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
from glob import glob

from matplotlib.lines import Line2D

plt.rcParams.update({'font.size': 10})

In [ ]:
def calculate_temp_variation(coefficients, model_name):

    bsplines = pd.read_csv('data/temperature_bsplines.csv')

    temp_coef = []
    for i, coefs in coefficients.iterrows():

        temp = np.linspace(10, 35, 200)

        if model_name == 'Climate(lag 1) + Year|region + Month|region':
            lag1_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag1{i}'] for i in range(1, 5))

            temp_variation = pd.DataFrame({
                'coef': lag1_terms,
                'temp': temp
            })
        elif model_name == 'Climate(lag 1-3) + Year|region + Month|region':
            lag1_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag1{i}'] for i in range(1, 5))
            lag2_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag2{i}'] for i in range(1, 5))
            lag3_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag3{i}'] for i in range(1, 5))

            temp_variation = pd.DataFrame({
                'coef': lag1_terms + lag2_terms + lag3_terms,
                'temp': temp
            })
        elif model_name in ['Year + Month + P(Climate)']:
            degree1_terms = sum(temp * coefs.loc[f'mean_2m_air_temp_degree1_lag{i}'] for i in range(1, 4))
            degree2_terms = sum((temp ** 2) * coefs.loc[f'mean_2m_air_temp_degree2_lag{i}'] for i in range(1, 4))
            degree3_terms = sum((temp ** 3) * coefs.loc[f'mean_2m_air_temp_degree3_lag{i}'] for i in range(1, 4))

            temp_variation = pd.DataFrame({
                'coef': degree1_terms + degree2_terms + degree3_terms,
                'temp': temp
            })
        else:
            lag1_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag1{i}'] for i in range(1, 5))
            lag2_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag2{i}'] for i in range(1, 5))
            lag3_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag3{i}'] for i in range(1, 5))
            lag4_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag4{i}'] for i in range(1, 5))
            lag5_terms = sum(bsplines[f'bs_{i}'] * coefs.loc[f'temp_bs_lag5{i}'] for i in range(1, 5))

            temp_variation = pd.DataFrame({
                'coef': lag1_terms + lag2_terms + lag3_terms + lag4_terms + lag5_terms,
                'temp': temp
            })

        temp_variation['model'] = i
        temp_coef.append(temp_variation)

    temp_coef = pd.concat(temp_coef)
    temp_coef_variation = temp_coef.groupby('model').apply(lambda x: x.set_index('temp')['coef'].diff()).reset_index().transpose()
    temp_coef_variation = temp_coef_variation.drop('model')

    quantiles = temp_coef_variation.quantile([0.025, 0.5, 0.975], axis=1).transpose().reset_index()
    quantiles['temp'] = quantiles['temp'].astype(float)

    return quantiles


In [ ]:
df = pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv')

df['mean_2m_air_temp_degree1'].quantile(0.01), df['mean_2m_air_temp_degree1'].quantile(0.99)

In [ ]:
models = [
  'Climate(lag 1-5)',
  'Climate(lag 1-5) + Year|region',
  'Climate(lag 1-5) + Month|region',
  'Year + Month + P(Climate)',
  'Climate(lag 1) + Year|region + Month|region',
  'Climate(lag 1-3) + Year|region + Month|region',
  'Climate(lag 1-5) + Year|region + Month|region',
  'Climate(lag 1-5) + Year|region + Month|region + Socio',
  'Climate(lag 1-5) + Year|region + Month|region + PriorCases',
  'Climate(lag 1-5) + Year|region + Month|region + SeroRepla',
  'Climate(lag 1-5) + Year|region + Month|region + Immunity',
  'Climate(lag 1-5) + Year|region + Month|region + PriorCases + SeroRepla + Socio + Immunity',
  'Climate(lag 1-5) + Socio',
  'Climate(lag 1-5) + PriorCases',
  'Climate(lag 1-5) + SeroRepla',
  'Climate(lag 1-5) + Immunity',
  'Climate(lag 1-5) + PriorCases + SeroRepla + Socio + Immunity'
]

model_ids = {
    'Climate(lag 1-5)': 'C15',
    'Climate(lag 1-5) + Month|region': 'C15-Mr',
    'Climate(lag 1-5) + PriorCases': 'C15-c(PC)',
    'Climate(lag 1-5) + SeroRepla': 'C15-c(SR)',
    'Climate(lag 1-5) + Socio': 'C15-c(S)',
    'Climate(lag 1-5) + Immunity': 'C15-c(IM)',
    'Year + Month': 'Y-M',
    'Month|region + PriorCases + SeroRepla + Socio + Immunity': 'Mr-c(PC,SR,S,IM)',
    'Climate(lag 1-5) + PriorCases + SeroRepla + Socio + Immunity': 'C15-c(PC,SR,S,IM)',
    'Year + Month + P(Climate)': 'Y-M-P(C)',
    'Climate(lag 1-5) + Year|region': 'C15-Yr',
    'Climate(lag 1) + Year|region + Month|region': 'C1-Yr-Mr',
    'Climate(lag 1-3) + Year|region + Month|region': 'C13-Yr-Mr',
    'Climate(lag 1-5) + Year|region + Month|region': 'C15-Yr-Mr',
    'Climate(lag 1-5) + Year|region + Month|region + SeroRepla': 'C15-Yr-Mr-c(SR)',
    'Climate(lag 1-5) + Year|region + Month|region + Immunity': 'C15-Yr-Mr-c(IM)',
    'Climate(lag 1-5) + Year|region + Month|region + Socio': 'C15-Yr-Mr-c(S)',
    'Climate(lag 1-5) + Year|region + Month|region + PriorCases': 'C15-Yr-Mr-c(PC)',
    'Climate(lag 1-5) + Year|region + Month|region + PriorCases + SeroRepla + Socio + Immunity': 'C15-Yr-Mr-c(PC,SR,S,IM)'
}

df = pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv')

fig, axarr = plt.subplots(3, 6, figsize=(12, 6), sharex=True, sharey=True)

for i, (model, ax) in enumerate(zip(models, axarr.flatten())):

    coefficients_model = pd.read_csv(f'data/{model}_coef_state_blockboot1000.csv')
    quantiles_model = calculate_temp_variation(coefficients_model, model)

    quantiles_model.plot(x='temp', y=0.5, ax=ax, color='k', legend=False)
    ax.fill_between(
        quantiles_model['temp'],
        quantiles_model[0.025],
        quantiles_model[0.975],
        color='k', alpha=0.2
    )


for i, (model, ax) in enumerate(zip(models, axarr.flatten())):
    model_tag = re.sub(r'[^A-Za-z0-9]+', '_', model)
    model_tag = re.sub(r'_+$', '', model_tag)  # strip trailing underscores

    ifiles = glob(f'results/*_{model_tag}/*_{model_tag}_fe_samples_temp.csv')
    if len(ifiles) > 0:
        ifile = ifiles[0]
        if os.path.exists(ifile):
            try:
                coefficients_models = pd.read_csv(ifile)
                quantiles_model = calculate_temp_variation(coefficients_models, model)
    
                quantiles_model.plot(x='temp', y=0.5, ax=ax, color='C1', legend=False)
                ax.fill_between(
                    quantiles_model['temp'],
                    quantiles_model[0.025],
                    quantiles_model[0.975],
                    color='C1', alpha=0.2
                )
            except Exception:
                pass
    
    axt = ax.twinx()
    sns.histplot(df['mean_2m_air_temp_degree1'], bins=30, color='k', alpha=0.2, ax=axt, stat='density', zorder=0)
    sns.kdeplot(df['mean_2m_air_temp_degree1'], color='k', lw=2, ax=axt)
    
    ax.axhline(0, color='k', ls='--')
    ax.set_ylim([-1.4, 0.5])
    ax.set_title(model_ids[model], fontsize=8)
    ax.axhline(-0.7, color='k', ls='-')
    ax.set_yticks([-0.5, -0.25, 0.0, 0.25, 0.5])
    axt.set_ylim(0, 0.8)
    axt.set_ylabel('')
    ax.set_xlabel('')
    ax.set_xlim(df['mean_2m_air_temp_degree1'].quantile(0.01), df['mean_2m_air_temp_degree1'].quantile(0.99))

    if i not in [5, 11]:
        axt.set_yticklabels([])
        axt.set_yticks([])
    else:
        axt.set_yticks([0.0, 0.1, 0.2, 0.3])
        

df_mexico = pd.read_csv('data/model_input_mexico_immunity_city.csv')

coefficients_model = pd.read_csv(f'data/main_coef_state_mexico_blockboot1000.csv')
quantiles_model = calculate_temp_variation(coefficients_model, model)

ax = axarr[-1, -1]
axt = ax.twinx()

quantiles_model.plot(x='temp', y=0.5, ax=ax, color='k', legend=False)
ax.fill_between(
    quantiles_model['temp'],
    quantiles_model[0.025],
    quantiles_model[0.975],
    color='k', alpha=0.2
)

coefficients_model = pd.read_csv(f'results/mexico/06_fe_samples_original_scale.csv')
quantiles_model = calculate_temp_variation(coefficients_model, model)

quantiles_model.plot(x='temp', y=0.5, ax=ax, color='C1', legend=False)
ax.fill_between(
    quantiles_model['temp'],
    quantiles_model[0.025],
    quantiles_model[0.975],
    color='C1', alpha=0.2
)

sns.histplot(df_mexico['mean_2m_air_temp_degree1'], bins=30, color='k', alpha=0.2, ax=axt, stat='density', zorder=0)
sns.kdeplot(df_mexico['mean_2m_air_temp_degree1'], color='k', lw=2, ax=axt)

ax.axhline(0, color='k', ls='--')
ax.set_ylim([-1.4, 0.5])
ax.set_title('Mexico C15-Yr-Mr', fontsize=8)
ax.axhline(-0.7, color='k', ls='-')
ax.set_yticks([-0.5, -0.25, 0.0, 0.25, 0.5])
axt.set_ylim(0, 0.8)
axt.set_ylabel('')
ax.set_xlabel('')
axt.set_yticks([0.0, 0.1, 0.2, 0.3])
ax.set_xlim(df['mean_2m_air_temp_degree1'].quantile(0.01), df['mean_2m_air_temp_degree1'].quantile(0.99))

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

legend_elements = []

legend_elements.append(Line2D([0], [0], color='k', lw=2.5, label='fixest'))
legend_elements.append(Line2D([0], [0], color='C1', lw=2.5, label='INLA'))

# Place legend on the figure level or first axis
fig.legend(handles=legend_elements, frameon=False, ncol=4, bbox_to_anchor=(0.585, 0.06))

axarr[0, 0].text(-0.6, -0.6, 'Temperature Coefficient \n Variation', transform=axarr[0, 0].transAxes, va='center', ha='center', rotation=90, fontsize=14)
axarr[0, -1].text(1.4, -0.6, 'Density', transform=axarr[0, -1].transAxes, va='center', ha='center', rotation=90, fontsize=14)
axarr[2, 3].text(-0.2, -0.6, r'Temperature ($^o$C)', transform=axarr[2, 3].transAxes, va='center', ha='center', fontsize=14)

plt.savefig('temp_coefficients.pdf', bbox_inches='tight')
#fig.delaxes(axarr[-1, -1])

In [ ]:
def calculate_prec_variation(coefficients, model_name):

    prec = np.linspace(0, 1200, 400)

    prec_coef = []
    for i, coefs in coefficients.iterrows():

        if model_name == 'Climate(lag 1) + Year|region + Month|region':
            lag1_terms = prec * coefs.loc[f'total_precipitation_lag1']

            prec_variation = pd.DataFrame({
                'coef': lag1_terms,
                'prec': prec
            })

        elif model_name in ['Climate(lag 1-3) + Year|region + Month|region', 'Year + Month + P(Climate)']:
            lag1_terms = prec * coefs.loc[f'total_precipitation_lag1']
            lag2_terms = prec * coefs.loc[f'total_precipitation_lag2']
            lag3_terms = prec * coefs.loc[f'total_precipitation_lag3']

            prec_variation = pd.DataFrame({
                'coef': lag1_terms + lag2_terms + lag3_terms,
                'prec': prec
            })

        else:
            lag1_terms = prec * coefs.loc[f'total_precipitation_lag1']
            lag2_terms = prec * coefs.loc[f'total_precipitation_lag2']
            lag3_terms = prec * coefs.loc[f'total_precipitation_lag3']
            lag4_terms = prec * coefs.loc[f'total_precipitation_lag4']
            lag5_terms = prec * coefs.loc[f'total_precipitation_lag5']

            prec_variation = pd.DataFrame({
                'coef': lag1_terms + lag2_terms + lag3_terms + lag4_terms + lag5_terms,
                'prec': prec
            })
        
        prec_variation['model'] = i
        prec_coef.append(prec_variation)

    prec_coef = pd.concat(prec_coef)
    prec_coef_variation = prec_coef.groupby('model').apply(lambda x: x.set_index('prec')['coef'].diff()).reset_index().transpose()
    prec_coef_variation = prec_coef_variation.drop('model')

    quantiles = prec_coef_variation.quantile([0.025, 0.5, 0.975], axis=1).transpose().reset_index()
    quantiles['prec'] = quantiles['prec'].astype(float)

    return quantiles

In [ ]:
models = [
  'Climate(lag 1-5)',
  'Climate(lag 1-5) + Year|region',
  'Climate(lag 1-5) + Month|region',
  'Year + Month + P(Climate)',
  'Climate(lag 1) + Year|region + Month|region',
  'Climate(lag 1-3) + Year|region + Month|region',
  'Climate(lag 1-5) + Year|region + Month|region',
  'Climate(lag 1-5) + Year|region + Month|region + Socio',
  'Climate(lag 1-5) + Year|region + Month|region + PriorCases',
  'Climate(lag 1-5) + Year|region + Month|region + SeroRepla',
  'Climate(lag 1-5) + Year|region + Month|region + Immunity',
  'Climate(lag 1-5) + Year|region + Month|region + PriorCases + SeroRepla + Socio + Immunity',
  'Climate(lag 1-5) + Socio',
  'Climate(lag 1-5) + PriorCases',
  'Climate(lag 1-5) + SeroRepla',
  'Climate(lag 1-5) + Immunity',
  'Climate(lag 1-5) + PriorCases + SeroRepla + Socio + Immunity'
]

model_ids = {
    'Climate(lag 1-5)': 'C15',
    'Climate(lag 1-5) + Month|region': 'C15-Mr',
    'Climate(lag 1-5) + PriorCases': 'C15-c(PC)',
    'Climate(lag 1-5) + SeroRepla': 'C15-c(SR)',
    'Climate(lag 1-5) + Socio': 'C15-c(S)',
    'Climate(lag 1-5) + Immunity': 'C15-c(IM)',
    'Year + Month': 'Y-M',
    'Month|region + PriorCases + SeroRepla + Socio + Immunity': 'Mr-c(PC,SR,S,IM)',
    'Climate(lag 1-5) + PriorCases + SeroRepla + Socio + Immunity': 'C15-c(PC,SR,S,IM)',
    'Year + Month + P(Climate)': 'Y-M-P(C)',
    'Climate(lag 1-5) + Year|region': 'C15-Yr',
    'Climate(lag 1) + Year|region + Month|region': 'C1-Yr-Mr',
    'Climate(lag 1-3) + Year|region + Month|region': 'C13-Yr-Mr',
    'Climate(lag 1-5) + Year|region + Month|region': 'C15-Yr-Mr',
    'Climate(lag 1-5) + Year|region + Month|region + SeroRepla': 'C15-Yr-Mr-c(SR)',
    'Climate(lag 1-5) + Year|region + Month|region + Immunity': 'C15-Yr-Mr-c(IM)',
    'Climate(lag 1-5) + Year|region + Month|region + Socio': 'C15-Yr-Mr-c(S)',
    'Climate(lag 1-5) + Year|region + Month|region + PriorCases': 'C15-Yr-Mr-c(PC)',
    'Climate(lag 1-5) + Year|region + Month|region + PriorCases + SeroRepla + Socio + Immunity': 'C15-Yr-Mr-c(PC,SR,S,IM)'
}

df = pd.read_csv('data/model_input_brazil_immunity_city_with_priorinf.csv')

fig, axarr = plt.subplots(3, 6, figsize=(12, 6), sharex=True, sharey=True)

for i, (model, ax) in enumerate(zip(models, axarr.flatten())):
    
    coefficients_model = pd.read_csv(f'data/{model}_coef_state_blockboot1000.csv')
    quantiles_model = calculate_prec_variation(coefficients_model, model)

    quantiles_model.plot(x='prec', y=0.5, ax=ax, color='k', legend=False)
    ax.fill_between(
        quantiles_model['prec'],
        quantiles_model[0.025],
        quantiles_model[0.975],
        color='k', alpha=0.2
    )

    axt = ax.twinx()
    sns.histplot(df['total_precipitation'], stat='density', color='gray', ax=axt, bins=np.linspace(0, 1200, 200))

    ax.axhline(0, color='k', ls='--')
    ax.set_xlim(df['total_precipitation'].min(), df['total_precipitation'].quantile(0.99))
    #ax.set_ylim([-0.16, 0.02])
    ax.set_title(model_ids[model], fontsize=8)    
    ax.axhline(-0.03, color='k', ls='-')
    ax.set_yticks([-0.02, 0.0, 0.02, 0.04])
    axt.set_ylim([0, 0.04])
    axt.set_ylabel('')
    ax.set_xlabel('')

    if i not in [5, 11]:
        axt.set_yticklabels([])
        axt.set_yticks([])
    else:
        axt.set_yticks([0, 0.005, 0.01])

for i, (model, ax) in enumerate(zip(models, axarr.flatten())):
    model_tag = re.sub(r'[^A-Za-z0-9]+', '_', model)
    model_tag = re.sub(r'_+$', '', model_tag)  # strip trailing underscores

    ifiles = glob(f'results/*_{model_tag}/*_fe_samples_precip.csv')
    print(model_ids[model])
    print(ifiles)
    if len(ifiles) > 0:
        ifile = ifiles[0]
        if os.path.exists(ifile):
            try:
                coefficients_models = pd.read_csv(ifile)
                quantiles_model = calculate_prec_variation(coefficients_models, model)
    
                quantiles_model.plot(x='prec', y=0.5, ax=ax, color='C1', legend=False)
                ax.fill_between(
                    quantiles_model['prec'],
                    quantiles_model[0.025],
                    quantiles_model[0.975],
                    color='C1', alpha=0.2
                )
            except Exception:
                pass

    
df_mexico = pd.read_csv('data/model_input_mexico_immunity_city.csv')

coefficients_model = pd.read_csv(f'data/main_coef_state_mexico_blockboot1000.csv')
quantiles_model = calculate_prec_variation(coefficients_model, model)

ax = axarr[-1, -1]
axt = ax.twinx()

quantiles_model.plot(x='prec', y=0.5, ax=ax, color='k', legend=False)
ax.fill_between(
    quantiles_model['prec'],
    quantiles_model[0.025],
    quantiles_model[0.975],
    color='k', alpha=0.2
)

coefficients_model = pd.read_csv(f'results/mexico/06_fe_samples_precip_original_scale.csv')
quantiles_model = calculate_prec_variation(coefficients_model, model)

quantiles_model.plot(x='prec', y=0.5, ax=ax, color='C1', legend=False)
ax.fill_between(
    quantiles_model['prec'],
    quantiles_model[0.025],
    quantiles_model[0.975],
    color='C1', alpha=0.2
)

sns.histplot(df_mexico['total_precipitation'], bins=30, color='k', alpha=0.2, ax=axt, stat='density', zorder=0)
sns.kdeplot(df_mexico['total_precipitation'], color='k', lw=2, ax=axt)

ax.axhline(0, color='k', ls='--')
ax.set_xlim(df_mexico['total_precipitation'].min(), df_mexico['total_precipitation'].quantile(0.99))
ax.set_ylim([-0.16, 0.02])
ax.set_title('Mexico C15-Yr-Mr', fontsize=8)
ax.axhline(-0.03, color='k', ls='-')
ax.set_yticks([-0.02, 0.0, 0.02, 0.04])
axt.set_ylim([0, 0.04])
axt.set_yticks([0, 0.005, 0.01])
axt.set_ylabel('')
ax.set_xlabel('')

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

legend_elements = []

legend_elements.append(Line2D([0], [0], color='k', lw=2.5, label='fixest'))
legend_elements.append(Line2D([0], [0], color='C1', lw=2.5, label='INLA'))

# Place legend on the figure level or first axis
fig.legend(handles=legend_elements, frameon=False, ncol=4, bbox_to_anchor=(0.585, 0.05))

axarr[0, 0].text(-0.6, -0.6, 'Precipitation Coefficient \n Variation', transform=axarr[0, 0].transAxes, va='center', ha='center', rotation=90, fontsize=14)
axarr[0, -1].text(1.4, -0.6, 'Density', transform=axarr[0, -1].transAxes, va='center', ha='center', rotation=90, fontsize=14)
axarr[2, 3].text(-0.2, -0.6, r'Precipitation (mm)', transform=axarr[2, 3].transAxes, va='center', ha='center', fontsize=14)

plt.savefig('precip_coefficients.pdf', bbox_inches='tight')

#fig.delaxes(axarr[-1, -1])